# ⚡ Módulo 12 - Notebook 02: Operaciones de columnas y Functions

## 📑 Filtrado, selección y manipulación avanzada

**Libro:** Saliendo de lo Pandito  
**Módulo:** 12 - PySpark Transformación Avanzada  
**Duración estimada:** 70 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Filtrar** datos con condiciones complejas  
✅ **Seleccionar** y renombrar columnas eficientemente  
✅ **Aplicar** funciones de agregación y ventana  
✅ **Usar** expresiones SQL en DataFrames  
✅ **Optimizar** operaciones de columnas

---

## 📋 Pre-requisitos

* ✅ Notebook 12_01 completado (Transformaciones básicas)
* ✅ Conocimiento de withColumn()
* ✅ Familiaridad con pyspark.sql.functions

---

## 📚 Contenido

1. Filtrado Avanzado (filter/where)
2. Selección y Renombrado de Columnas
3. Expresiones SQL en DataFrames
4. Funciones de Agregación
5. Funciones de Ventana (Window Functions)
6. Caso Integrador: Análisis de Ventas Complejo

---

## 💡 Por qué importa

**Operaciones de columnas son fundamentales:**

* 🔍 **Filtrado:** Reducir volumen de datos procesados
* 📊 **Agregación:** Resumir millones de registros
* 📝 **Expresiones SQL:** Aprovechar conocimiento SQL
* ⚡ **Optimización:** Predicate pushdown automático

**La base de todo pipeline ETL**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (SPARK DataFrame)
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros totales: {df.count():,}")
    print(f"   🏛️ Particiones: {df.rdd.getNumPartitions()}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    # Mostrar muestra de datos
    print(f"\n📊 Muestra de datos (primeras 5 filas):")
    df.show(5, truncate=False)
    
    print(f"\n🎯 Este notebook aplicará:")
    print(f"   • Filtrado complejo con múltiples condiciones")
    print(f"   • Selección y renombrado de columnas")
    print(f"   • Agregaciones por grupos")
    print(f"   • Funciones de ventana (ranking, cumulative)")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Operaciones Avanzadas de Columnas

### 🔍 Filtrado Avanzado (filter / where)

**Sintaxis:**
```python
# Ambos son equivalentes
df.filter(condicion)
df.where(condicion)
```

**Formas de expresar condiciones:**

**1️⃣ String SQL:**
```python
df.filter("ventas > 100000")
df.filter("ventas > 100000 AND zona = 'Centro'")
```

**2️⃣ Expresión con F.col():**
```python
df.filter(F.col("ventas") > 100000)
df.filter((F.col("ventas") > 100000) & (F.col("zona") == "Centro"))
```

**3️⃣ Múltiples condiciones:**
```python
# AND: &
df.filter((F.col("ventas") > 100000) & (F.col("margen") > 0.20))

# OR: |
df.filter((F.col("zona") == "Centro") | (F.col("zona") == "Norte"))

# NOT: ~
df.filter(~F.col("zona").isin(["Sur", "Este"]))
```

**⚠️ IMPORTANTE:** Usar `&` y `|`, NO `and` y `or` de Python.

---

### 📑 Selección y Renombrado de Columnas

**Seleccionar columnas:**
```python
# Método 1: Lista de strings
df.select("sucursal", "ventas", "fecha")

# Método 2: F.col()
df.select(F.col("sucursal"), F.col("ventas"))

# Método 3: Con transformaciones
df.select(
    F.col("sucursal"),
    (F.col("ventas") * 1.21).alias("ventas_con_iva")
)
```

**Renombrar columnas:**
```python
# Método 1: alias()
df.select(
    F.col("sucursal").alias("tienda"),
    F.col("ventas").alias("monto")
)

# Método 2: withColumnRenamed()
df.withColumnRenamed("sucursal", "tienda")

# Método 3: Renombrar múltiples
df.toDF("col1", "col2", "col3")  # Renombra todas
```

**Drop columnas:**
```python
df.drop("columna_innecesaria", "otra_columna")
```

---

### 📊 Agregaciones

**Agregación simple:**
```python
df.agg(
    F.sum("ventas").alias("ventas_totales"),
    F.avg("ventas").alias("ventas_promedio"),
    F.count("*").alias("cantidad_registros")
)
```

**Agregación por grupo:**
```python
df.groupBy("zona").agg(
    F.sum("ventas").alias("ventas_totales"),
    F.avg("ventas").alias("ventas_promedio"),
    F.count("*").alias("sucursales")
)
```

**Múltiples dimensiones:**
```python
df.groupBy("zona", "mes").agg(
    F.sum("ventas").alias("ventas_totales")
).orderBy("zona", "mes")
```

---

### 📝 Expresiones SQL en DataFrames

**selectExpr() - SQL directo:**
```python
df.selectExpr(
    "sucursal",
    "ventas * 1.21 AS ventas_con_iva",
    "CASE WHEN ventas > 100000 THEN 'Alto' ELSE 'Bajo' END AS categoria"
)
```

**expr() - Expresiones complejas:**
```python
from pyspark.sql.functions import expr

df.withColumn("margen_pct", expr("(ventas - costo) / ventas * 100"))
```

---

### 📊 Funciones de Ventana (Window Functions)

**Concepto:** Operaciones sobre un "grupo" de filas relacionadas.

**Definir ventana:**
```python
from pyspark.sql.window import Window

# Ventana particionada por zona, ordenada por ventas
window_spec = Window.partitionBy("zona").orderBy(F.desc("ventas"))
```

**Ranking:**
```python
df.withColumn(
    "ranking_por_zona",
    F.row_number().over(window_spec)
)
```

**Acumulados:**
```python
df.withColumn(
    "ventas_acumuladas",
    F.sum("ventas").over(
        Window.partitionBy("zona").orderBy("fecha")
    )
)
```

**Lag/Lead (valores previos/siguientes):**
```python
# Ventas del mes anterior
df.withColumn(
    "ventas_mes_anterior",
    F.lag("ventas", 1).over(
        Window.partitionBy("sucursal").orderBy("fecha")
    )
)

# Calcular variación
df.withColumn(
    "variacion",
    F.col("ventas") - F.col("ventas_mes_anterior")
)
```

---

### 💼 Caso de Uso: Top 3 Sucursales por Zona

```python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Agregar ventas por sucursal
df_agg = df.groupBy("zona", "sucursal").agg(
    F.sum("ventas").alias("ventas_totales")
)

# 2. Definir ventana por zona
window = Window.partitionBy("zona").orderBy(F.desc("ventas_totales"))

# 3. Agregar ranking
df_ranked = df_agg.withColumn("ranking", F.row_number().over(window))

# 4. Filtrar top 3
top3 = df_ranked.filter(F.col("ranking") <= 3)

top3.show()
```

**Resultado:**
```
+-------+-----------+--------------+-------+
|  zona |  sucursal |ventas_totales|ranking|
+-------+-----------+--------------+-------+
|Centro |Sucursal A |     5,500,000|      1|
|Centro |Sucursal B |     4,200,000|      2|
|Centro |Sucursal C |     3,800,000|      3|
| Norte |Sucursal D |     4,100,000|      1|
| Norte |Sucursal E |     3,900,000|      2|
| Norte |Sucursal F |     3,500,000|      3|
+-------+-----------+--------------+-------+
```

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("📑 OPERACIONES DE COLUMNAS Y FUNCTIONS")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • filter/where - Filtrado avanzado")
print("  • select/selectExpr - Selección y proyección")
print("  • groupBy + agg - Agregaciones")
print("  • Window Functions - Ranking y acumulados")

print("\n📖 Funciones clave:")
print("  - df.filter(condicion)")
print("  - df.select(F.col('col').alias('nuevo'))")
print("  - df.groupBy('col').agg(F.sum('val'))")
print("  - Window.partitionBy('col').orderBy('val')")
print("  - F.row_number(), F.rank(), F.lag(), F.lead()")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 📑 Filtrado, agregación y window functions con datos reales

### 🔍 Filtrado distribuido en Los Andes Market

Con el Spark DataFrame de ventas podemos filtrar y agregar a escala distribuida:

```python
# Filtrado con múltiples condiciones (AND, OR)
df.filter((F.col("ventas") > 100000) & (F.col("zona") == "Centro Comercial"))

# Agregación por zona
df.groupBy("zona").agg(F.sum("ventas").alias("total"), F.avg("ventas").alias("promedio"))
```

---

### 🏆 Window Functions: Ranking y comparativas

Las window functions permiten calcular rankings y comparativas sin colapsar filas:

```python
from pyspark.sql.window import Window

# Ranking de sucursales por ventas dentro de cada zona
w = Window.partitionBy("zona").orderBy(F.desc("ventas"))
df.withColumn("ranking_zona", F.row_number().over(w))

# Variación mes anterior
df.withColumn("ventas_anterior", F.lag("ventas", 1).over(w_orden))
```

---

### 💡 Preguntas de negocio
* ¿Qué sucursal vende más dentro de su zona?
* ¿Cómo varían las ventas respecto al mes anterior?
* ¿Cuál es el promedio móvil de 3 meses por sucursal?

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("📑 FILTRADO, AGREGACIÓN Y WINDOW FUNCTIONS CON DATOS REALES")
print("="*70)

if USAR_DATOS_REALES and df is not None:
    df = df.withColumn("año", F.year(F.col("fecha")))
    df = df.withColumn("mes", F.month(F.col("fecha")))

    print("\n1️⃣  FILTRADO AVANZADO: Múltiples condiciones")
    print("-"*70)

    # AND: ventas altas en zona específica
    df_alto_centro = df.filter((F.col("ventas") > 100000) & (F.col("zona").contains("Centro")))
    print(f"\n   Ventas > 100k en zona Centro: {df_alto_centro.count()} registros")
    df_alto_centro.select("sucursal_nombre", "zona", "ventas", "fecha").show(5, truncate=30)

    # OR: zonas residenciales o comerciales
    df_comerciales = df.filter(
        (F.col("zona") == "Zona Residencial") | (F.col("zona") == "Corredor Comercial")
    )
    print(f"\n   Zonas Residencial + Corredor: {df_comerciales.count()} registros")

    # NOT: excluir zona
    df_sin_residencial = df.filter(~F.col("zona").contains("Residencial"))
    print(f"   Excluyendo Residencial: {df_sin_residencial.count()} registros")

    print("\n" + "="*70)
    print("\n2️⃣  AGREGACIÓN: Ventas por zona y año")
    print("-"*70)

    agg_zona = (df
        .groupBy("zona", "año")
        .agg(
            F.round(F.sum("ventas"), 0).alias("ventas_totales"),
            F.round(F.avg("ventas"), 0).alias("ventas_promedio"),
            F.count("*").alias("registros"),
            F.round(F.max("ventas"), 0).alias("venta_max"),
            F.round(F.min("ventas"), 0).alias("venta_min")
        )
        .orderBy("zona", "año")
    )
    agg_zona.show(20, truncate=30)

    print("\n" + "="*70)
    print("\n3️⃣  WINDOW: Ranking de sucursales por zona")
    print("-"*70)

    w_rank = Window.partitionBy("zona").orderBy(F.desc("ventas"))
    df_ranked = df.withColumn("ranking_zona", F.row_number().over(w_rank))

    print("\n   Top 3 sucursales por zona (ranking):")
    df_ranked.filter(F.col("ranking_zona") <= 3).select(
        "zona", "sucursal_nombre", "ventas", "ranking_zona", "fecha"
    ).orderBy("zona", "ranking_zona").show(20, truncate=30)

    print("\n" + "="*70)
    print("\n4️⃣  WINDOW: lag() para comparar con mes anterior")
    print("-"*70)

    w_time = Window.partitionBy("sucursal_nombre").orderBy("fecha")
    df_lag = (df
        .withColumn("ventas_anterior", F.lag("ventas", 1).over(w_time))
        .withColumn("variacion", F.round(
            (F.col("ventas") - F.col("ventas_anterior")) / F.col("ventas_anterior") * 100, 2
        ))
    )
    print("\n   Ventas vs mes anterior (% variación):")
    df_lag.select("sucursal_nombre", "fecha", "ventas", "ventas_anterior", "variacion").show(10, truncate=30)
    print("   💡 lag(1) toma el valor de la fila anterior en la ventana")

    print("\n" + "="*70)
    print("\n5️⃣  WINDOW: Promedio móvil de 3 meses")
    print("-"*70)

    w_rolling = (Window
        .partitionBy("sucursal_nombre")
        .orderBy("fecha")
        .rowsBetween(-2, 0)  # 3 meses: actual + 2 anteriores
    )
    df_rolling = df.withColumn("promedio_movil_3m", F.round(F.avg("ventas").over(w_rolling), 0))
    print("\n   Promedio móvil 3 meses por sucursal:")
    df_rolling.select("sucursal_nombre", "fecha", "ventas", "promedio_movil_3m").show(10, truncate=30)

    print("\n" + "="*70)
    print("\n6️⃣  SELECTEXPR: SQL directo en DataFrames")
    print("-"*70)

    df_sql = df.selectExpr(
        "sucursal_nombre",
        "zona",
        "ventas",
        "round(ventas * 1.21, 2) as ventas_iva",
        "case when ventas > 100000 then 'Alto' else 'Bajo' end as categoria"
    )
    df_sql.show(10, truncate=30)
    print("\n   💡 selectExpr permite usar sintaxis SQL directamente")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')

## 🎓 Conclusiones del notebook 12_02

### ✅ Lo que aprendiste

1. **Filtrado avanzado (filter/where):**
   - `df.filter(F.col('ventas') > 100000)` — expresión con F.col
   - `df.filter("ventas > 100000")` — string SQL directo
   - `&` (AND), `|` (OR), `~` (NOT) — NO usar `and`/`or` de Python
   - `F.col('zona').isin(['Centro', 'Norte'])` — filtro por lista

2. **Selección y renombrado de columnas:**
   - `df.select('col1', 'col2')` — seleccionar columnas
   - `F.col('ventas').alias('monto')` — renombrar con alias
   - `df.withColumnRenamed('old', 'new')` — renombrar sin transformar
   - `df.drop('col_innecesaria')` — eliminar columnas

3. **Expresiones SQL en DataFrames:**
   - `df.selectExpr("ventas * 1.21 AS ventas_con_iva")` — SQL directo
   - `F.expr("(ventas - costo) / ventas * 100")` — expresión compleja
   - Útil cuando el equipo conoce SQL pero no la API de PySpark

4. **Funciones de agregación:**
   - `df.groupBy('zona').agg(F.sum('ventas'), F.avg('ventas'))`
   - Múltiples métricas en una sola llamada con `.alias()`
   - `df.agg(...)` para agregación global sin groupBy

5. **Window Functions:**
   - `Window.partitionBy('zona').orderBy(F.desc('ventas'))` — define la ventana
   - `F.row_number().over(window)` — ranking secuencial
   - `F.sum('ventas').over(window)` — acumulado móvil
   - `F.lag('ventas', 1).over(window)` — valor del período anterior
   - `F.lead('ventas', 1).over(window)` — valor del período siguiente

---

### 🎯 Reglas de Oro

👉 **Regla #1: Usar & | ~ NO and or not**
```python
# MALO: 'and'/'or' de Python no funcionan en Spark
df.filter(F.col('ventas') > 100000 and F.col('zona') == 'Centro')  # 💥 Error

# BUENO: & | ~ son los operadores de Spark
df.filter((F.col('ventas') > 100000) & (F.col('zona') == 'Centro'))  # ✅
# ⚠️ Paréntesis obligatorios alrededor de cada condición
```

👉 **Regla #2: alias() en cada columna calculada**
```python
# MALO: sin alias, columnas con nombres autogenerados incomprensibles
df.select(F.col('ventas') * 1.21, F.col('costo') / F.col('ventas'))
# Resultado: (ventas * 1.21), (costo / ventas)

# BUENO: alias descriptivos
df.select(
    (F.col('ventas') * 1.21).alias('ventas_con_iva'),
    (F.col('costo') / F.col('ventas')).alias('margen_pct')
)
```

👉 **Regla #3: Window.partitionBy antes de orderBy**
```python
# MALO: orderBy sin partitionBy = ventana global (ranking de TODOS)
window = Window.orderBy(F.desc('ventas'))
df.withColumn('ranking', F.row_number().over(window))  # ranking global

# BUENO: partitionBy para ranking por grupo
window = Window.partitionBy('zona').orderBy(F.desc('ventas'))
df.withColumn('ranking', F.row_number().over(window))  # ranking por zona
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Filtrar con condición simple | `df.filter(F.col('x') > 100)` |
| Filtrar con múltiples condiciones | `df.filter((cond1) & (cond2))` |
| Filtrar por lista de valores | `F.col('x').isin(['a', 'b'])` |
| Seleccionar columnas | `df.select('col1', 'col2')` |
| Renombrar una columna | `df.withColumnRenamed('old', 'new')` |
| Columna calculada con alias | `F.col('x').alias('nuevo')` |
| SQL directo en DataFrame | `df.selectExpr('x * 1.21 AS iva')` |
| Agregación por grupo | `df.groupBy('col').agg(F.sum('val'))` |
| Ranking por grupo | `Window.partitionBy('g').orderBy(F.desc('v'))` |
| Acumulado por grupo | `F.sum('v').over(Window.partitionBy('g').orderBy('fecha'))` |
| Valor del período anterior | `F.lag('v', 1).over(Window.partitionBy('g').orderBy('fecha'))` |
| Valor del período siguiente | `F.lead('v', 1).over(Window.partitionBy('g').orderBy('fecha'))` |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>📑 ¡Operaciones de columnas y functions dominadas!</h3>
  <p><i>"filter + select + groupBy + window = el 90% del análisis con Spark DataFrames."</i></p>
</div>